# 06 — Compute customer and daily business metrics

**Student exercise**

Complete the tasks in order. Each task includes an expected result, three hints, and a check. All sample data and setup code are included in this notebook.

## 1. Spark configuration

Start a fresh kernel. Use Spark 3.5.x / Scala 2.12 and Java 17. Set `STUDENT_ID` to a lowercase name containing letters, numbers or underscores. Set `SPARK_MASTER=local[2]` for a local Spark session.

In [ ]:
import os
import socket
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

master_url = os.environ.get("SPARK_MASTER", f"spark://{socket.gethostname()}:7077")
STUDENT_ID = "student01"
assert STUDENT_ID and STUDENT_ID[0].islower() and all(c in "abcdefghijklmnopqrstuvwxyz0123456789_" for c in STUDENT_ID)

spark = (
    SparkSession.builder
    .appName("Exercise-06")
    .master(master_url)
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.shuffle.partitions", "2")
    .config("spark.redaction.regex", "(?i)secret|password|token|access[.]?key|credential")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

## 2. Sample data

Run this cell, inspect the records, and use the supplied variables in the tasks.

In [ ]:
orders = spark.createDataFrame([
    (101,1,40.0,'2026-09-01'), (102,2,50.0,'2026-09-01'),
    (103,1,30.0,'2026-09-02'), (104,3,20.0,'2026-09-02')],
    'id INT, customer_id INT, amount DOUBLE, order_date STRING')
orders.show()

## 3. Result checks

This small function checks output values and duplicate counts. It is included here so the notebook can run on its own.

In [ ]:
def check_frame(df, columns, expected):
    assert df is not None, "Complete the task before running the check"
    assert set(columns).issubset(df.columns), "Missing output columns"
    def normalize(row):
        return tuple(round(value, 6) if isinstance(value, float) else value for value in row)
    actual = sorted(repr(normalize(row)) for row in df.select(*columns).collect())
    wanted = sorted(repr(normalize(row)) for row in expected)
    assert actual == wanted, f"Expected {wanted}, got {actual}"
    print("PASS:", columns)

## Task 1: Customer counts, sums and averages

Create customer_metrics with customer_id, order_count, revenue and avg_order.

**Expected result:** Customer revenues: 70,50,20.

In [ ]:
customer_metrics=None  # TODO: groupBy and aggregate

**Hint 1:** count(*) counts rows; count(column) skips null values in that column.

**Hint 2:** Give every aggregation an explicit alias.

**Hint 3:** orders.groupBy("customer_id").agg(F.count("*"), F.sum(...), F.avg(...)).

### Check

In [ ]:
check_frame(customer_metrics,['customer_id','order_count','revenue','avg_order'],[(1,2,70.0,35.0),(2,1,50.0,50.0),(3,1,20.0,20.0)])

## Task 2: SQL HAVING filter

Create valuable using SQL: customers with at least two orders.

**Expected result:** Only customer 1.

In [ ]:
valuable=None  # TODO: temporary view, GROUP BY, HAVING

**Hint 1:** WHERE filters input rows; HAVING filters aggregate groups.

**Hint 2:** Use GROUP BY customer_id and HAVING COUNT(*)>=2.

**Hint 3:** SELECT key, aggregates FROM view GROUP BY key HAVING ... .

### Check

In [ ]:
check_frame(valuable,['customer_id','order_count','revenue'],[(1,2,70.0)])

## Task 3: Daily reconciliation

Group by order_date into daily with order_count and revenue. Reconcile its sum to the original.

**Expected result:** 2 orders per day, revenues 90 and 50.

In [ ]:
daily=None  # TODO: groupBy date and sum

**Hint 1:** Aggregation should conserve the total unless rows are filtered or multiplied.

**Hint 2:** Both daily and original totals should equal 140.

**Hint 3:** Use agg(sum(...)).first()[0] only because the check returns one tiny aggregate row.

### Check

In [ ]:
check_frame(daily,['order_date','order_count','revenue'],[('2026-09-01',2,90.0),('2026-09-02',2,50.0)])
assert daily.agg(F.sum('revenue')).first()[0]==orders.agg(F.sum('amount')).first()[0]

## More practice

Calculate distinct active customers; add a null amount and compare count(*) with count(amount).

## Common mistakes

Do not confuse row count with distinct customer count. Use decimal monetary types in real financial models.

## Finish

Stop Spark before starting another notebook. For write exercises, run setup again before repeating append tasks. S3 data remains available after stopping Spark.

In [ ]:
spark.stop()